# Aula 6 – Junção de Tabelas e Cruzamento de Dados

Nesta aula vamos:
1. Abrir as **Tabelas 5** (desocupação por sexo em 2012 e 2026) e a **Tabela 1.1.1** (horas de afazeres domésticos, 2022)
2. Juntar as duas Tabelas 5 lado a lado com `merge`
3. Cruzar essa tabela combinada com a Tabela 1.1.1 do IBGE

In [ ]:
import pandas as pd

## 1. Caminhos dos arquivos

In [ ]:
path_111  = "Tabela 1.1.1.xls"
path_2012 = "Tabela5-sem_emprego_2012.csv"
path_2026 = "Tabela5-sem_emprego_2026.csv"

## 2. Leitura dos arquivos

### 2a. Tabela 5 – 2012 e 2026

In [ ]:
t2012 = pd.read_csv(path_2012, decimal=",", encoding="UTF-8", sep=";")
t2026 = pd.read_csv(path_2026, decimal=",", encoding="UTF-8", sep=";")

print("=== Tabela 5 – 2012 ===")
print(t2012.head())
print("\n=== Tabela 5 – 2026 ===")
print(t2026.head())

### 2b. Tabela 1.1.1 – Horas de afazeres domésticos (IBGE, 2022)

O `.xls` do IBGE mistura **título**, cabeçalho em várias linhas e **notas no rodapé**.  
Cortamos isso e deixamos só: **nomes das colunas + números** (mesmo tratamento da Aula 2).

In [ ]:
# Lê sem usar a 1ª linha como nome de coluna (senão o título vira "coluna")
bruto = pd.read_excel(path_111, engine="xlrd", header=None)

# Linhas 0–7: título/cabeçalho  |  8–40: resultados  |  41+: fonte e notas
df_111 = bruto.iloc[8:41].copy()
df_111.columns = [
    "Estado",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
df_111 = df_111.reset_index(drop=True)

# Converte horas para número
for col in df_111.columns[1:]:
    df_111[col] = pd.to_numeric(df_111[col], errors="coerce")

print("Colunas:", df_111.columns.tolist())
df_111.head()

## 3. Juntando as duas Tabelas 5 (merge)

Unimos 2012 e 2026 pela chave `["Sigla", "Código", "Estado"]`.  
O sufixo `_2012` / `_2026` distingue as colunas de cada ano.

In [ ]:
comp = t2012.merge(
    t2026,
    on=["Sigla", "Código", "Estado"],
    how="inner",
    suffixes=("_2012", "_2026")
)

print("Linhas:", len(comp))
comp.head()

## 4. Cruzando com a Tabela 1.1.1 (horas de afazeres domésticos)

A Tabela 1.1.1 usa o **nome completo do estado** na coluna `Estado`.  
A Tabela 5 também usa o nome completo, então podemos cruzar diretamente.  

> **Atenção:** a Tabela 1.1.1 inclui linhas de **Grandes Regiões** (Norte, Nordeste…) e do **Brasil** que não existem na Tabela 5.  
> Usamos `how="inner"` para manter apenas os **estados** que aparecem nas duas tabelas.

In [ ]:
final = comp.merge(
    df_111,
    on="Estado",
    how="inner"
)

print("Linhas no cruzamento final:", len(final))
print("Colunas:", final.columns.tolist())
final.head()

## 5. Análise rápida

Exemplo: estado com **maior proporção de mulheres desocupadas em 2026**  
e o correspondente **número médio de horas de afazeres domésticos das mulheres**.

In [ ]:
col_mulher_2026 = [c for c in final.columns if "mulher" in c.lower() and "2026" in c][0]
col_mulher_2012 = [c for c in final.columns if "mulher" in c.lower() and "2012" in c][0]

print(f"Coluna desocupação mulheres 2026: {col_mulher_2026}")
print(f"Coluna desocupação mulheres 2012: {col_mulher_2012}")
idx_max = final[col_mulher_2026].idxmax()
estado_max = final.loc[idx_max]

print(f"\nEstado com maior % de mulheres desocupadas em 2026: {estado_max['Estado']}")
print(f"  Desocupação mulheres 2026: {estado_max[col_mulher_2026]:.1f}%")
print(f"  Desocupação mulheres 2012: {estado_max[col_mulher_2012]:.1f}%")
print(f"  Horas semanais afazeres – mulheres brancas: {estado_max['mulher_branca']:.2f}")
print(f"  Horas semanais afazeres – mulheres pretas/pardas: {estado_max['mulher_preta_parda']:.2f}")

## 6. Variação de 2012 para 2026

Criamos uma coluna de **diferença** (2026 − 2012) para mulheres desocupadas.

In [ ]:
final["variacao_mulher"] = final[col_mulher_2026] - final[col_mulher_2012]


aumentou = final[final["variacao_mulher"] > 0][["Estado", col_mulher_2012, col_mulher_2026, "variacao_mulher"]]
print(f"Estados onde a % de mulheres desocupadas aumentou de 2012 para 2026: {len(aumentou)}")
aumentou.sort_values("variacao_mulher", ascending=False)